# 🌀 NeuroForge CFD — AirfRANS training on Colab GPU

**Self-correcting, geometry-native AI CFD.** This notebook trains the NeuroForge
engine on the [AirfRANS](https://airfrans.readthedocs.io) dataset (incompressible
RANS over NACA airfoils) using a Colab GPU, then runs the self-correcting solver
(predict → check physics residuals → estimate uncertainty → **Neural Residual
Iteration** → optional classical fallback) and visualises the results.

**Pipeline:** install → (optional) mount Drive → download AirfRANS → train FNO +
corrector on GPU → evaluate field errors & Cl/Cd → self-correcting solve → plots.

> Runtime → *Change runtime type* → **GPU** (T4/L4/A100) before running.


## 0 · Check the GPU


In [ ]:
!nvidia-smi -L || echo 'No GPU — set Runtime > Change runtime type > GPU'


## 1 · Get the code & install

Set `REPO_URL` to your GitHub repo (push the local repo first), **or** upload the
project and point `REPO_DIR` at it, **or** mount Drive (section 2) and clone/copy there.


In [ ]:
import os

REPO_URL = 'https://github.com/ali-kin4/neuroforge-cfd.git'  # your repo
REPO_DIR = 'neuroforge-cfd'

if not os.path.isdir(REPO_DIR):
    rc = os.system(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')
    if rc != 0:
        raise RuntimeError(
            'Clone failed. Either push your repo and set REPO_URL, or upload the '
            'project folder and set REPO_DIR to its path (e.g. a Drive path).')
%cd {REPO_DIR}
# Editable install + the AirfRANS data extra. (threadpoolctl etc. come as deps.)
!pip install -q -e '.[data]'
print('installed NeuroForge CFD')


## 2 · (Optional) Mount Google Drive

Caching the rasterised dataset and checkpoints on Drive makes later sessions
instant. Skip this cell to keep everything in the ephemeral Colab disk.


In [ ]:
USE_DRIVE = True  # set False to use local Colab storage only

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/neuroforge_cfd'
else:
    BASE = '/content/neuroforge_runs'

DATA_ROOT = os.path.join(BASE, 'data')      # AirfRANS download/extract
CACHE_DIR = os.path.join(BASE, 'cache')     # cached rasterised pairs
CKPT_DIR  = os.path.join(BASE, 'checkpoints')
for d in (DATA_ROOT, CACHE_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)
print('BASE =', BASE)


## 3 · Threads + imports

NeuroForge caps math-library threads to 1 on import (it fixes a pathological
oversubscription on small/odd hosts). On a healthy Colab CPU we *want* a few
threads for data rasterisation, so we override **before** importing `neuroforge`.


In [ ]:
import os
n = str(min(4, os.cpu_count() or 2))
os.environ['OMP_NUM_THREADS'] = n
os.environ['OPENBLAS_NUM_THREADS'] = n
os.environ['MKL_NUM_THREADS'] = n

import torch, numpy as np, matplotlib.pyplot as plt
import neuroforge as nf
from neuroforge.core.config import Config, DataConfig, ModelConfig
from neuroforge.train import train_recipe
from neuroforge.data.airfrans_loader import load_airfrans, download_airfrans
from neuroforge.solver.engine import NeuroForgeEngine, Predictor
from neuroforge.viz.plots import overview_figure, plot_field, plot_cp, plot_convergence, plot_trust

print('neuroforge', nf.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## 4 · Download AirfRANS

The dataset is a few GB; the download/extract runs once (and is skipped if it
already exists under `DATA_ROOT`). Tasks: `scarce` (small), `full`, `reynolds`, `aoa`.


In [ ]:
TASK = 'full'   # 'full' = 800 train / 200 test. Use 'scarce' for a quicker pass.
download_airfrans(DATA_ROOT)   # downloads once (a few GB); skipped if present
print('AirfRANS ready under', DATA_ROOT)


## 5 · Configure & train (GPU)

A **stronger first-run** setup for a single GPU: a wider/deeper FNO (width 64,
5 layers, 24 Fourier modes) trained on 400 airfoils for 120 epochs, plus a
30-epoch residual-driven corrector. Loss = data + physics-residual + no-slip BC.

The **first** run rasterises + caches the data to `CACHE_DIR` (one-time, a few
minutes); later runs reuse the cache. Rough budget on a T4: ~15-25 min total;
much faster on an A100. **Too slow or out-of-memory?** Lower `n_train` (e.g. 200),
`batch_size` (e.g. 6), `width`/`modes`, or switch `TASK='scarce'`.


In [ ]:
cfg = Config()
cfg.data = DataConfig(
    source='airfrans', root=DATA_ROOT, task=TASK, resolution=128,
    n_train=400, n_val=120, batch_size=10, num_workers=2, cache_dir=CACHE_DIR,
)
# A solid first-run backbone for a single GPU (T4 / L4 / A100).
cfg.model = ModelConfig(name='fno', width=64, n_layers=5, modes=24)
cfg.train.epochs = 120
cfg.train.lr = 8e-4
cfg.train.weight_decay = 1e-5
cfg.train.physics_weight = 0.1
cfg.train.bc_weight = 0.1
cfg.train.grad_clip = 1.0
cfg.train.amp = True          # GPU mixed precision (bf16 on A100, fp16 else); no-op on CPU
cfg.train.device = 'auto'      # -> cuda on Colab
cfg.train.log_every = 50

ckpt_path = os.path.join(CKPT_DIR, f'airfrans_{TASK}_fno_w64.pt')
result = train_recipe(cfg, download=False, corrector_epochs=30, out=ckpt_path)
print('\nval rel-L2:', {k: round(result['val_errors'][k], 3)
      for k in ('u','v','p','speed') if k in result['val_errors']})


## 6 · Training curves & metrics


In [ ]:
h = result['history']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogy(h['train_loss'], label='train'); ax[0].semilogy(h['val_loss'], label='val')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('composite loss'); ax[0].legend(); ax[0].set_title('training')
if result['corrector_history']:
    ax[1].semilogy(result['corrector_history']['corrector_loss'])
    ax[1].set_xlabel('epoch'); ax[1].set_ylabel('corrector loss'); ax[1].set_title('corrector')
plt.tight_layout(); plt.show()
print('params:', f"{result['n_params']:,}", '| device:', result['device'],
      '| infer:', round(result['val_errors'].get('infer_ms_per_sample', 0), 2), 'ms/sample')


## 7 · The self-correcting engine on a held-out airfoil

Load a few validation cases (with ground-truth fields), run the full
**predict → check → Neural Residual Iteration** loop, and inspect the residual
history (guaranteed non-increasing by the backtracking acceptance test) plus the
uncertainty/trust maps and the predicted Cl/Cd.


In [ ]:
val_pairs = load_airfrans(DATA_ROOT, task=TASK, train=False, resolution=128,
                          limit=12, cache_dir=CACHE_DIR)
engine = NeuroForgeEngine.from_checkpoint(ckpt_path)   # CPU inference (fast)

case, gt = val_pairs[0]
res = engine.solve(case, max_iters=5)
print('case:', case.name)
print('residual history:', [round(s['residual_norm'], 4) for s in res.history])
print('metrics:', {k: round(v, 4) for k, v in res.metrics.items()
                   if k in ('cl','cd','cm','residual_norm','trust_mean')})
fig = overview_figure(res); plt.show()


## 8 · Prediction vs. ground truth

Side-by-side of predicted and CFD-reference fields on the held-out case, plus the
surface pressure coefficient (Cp).


In [ ]:
from neuroforge.core.types import SolveResult
pred_field = res.field

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, key in enumerate(['speed', 'p', 'u']):
    plot_field(pred_field, key=key, ax=axes[0, col]); axes[0, col].set_title(f'pred {key}')
    plot_field(gt,         key=key, ax=axes[1, col]); axes[1, col].set_title(f'CFD {key}')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
plot_cp(pred_field, case, ax=ax, ref=gt); ax.set_title('Cp: prediction vs CFD'); plt.show()


## 9 · Aggregate Cl/Cd over the validation set


In [ ]:
from neuroforge.physics.metrics import force_coefficients
pred = Predictor(result['model'], result['normalizer'],
                 device='cuda' if torch.cuda.is_available() else 'cpu')
rows = []
for case, gt in val_pairs:
    pf = pred.predict(case)
    fp, fg = force_coefficients(pf, case), force_coefficients(gt, case)
    rows.append((fp['cl'], fg['cl'], fp['cd'], fg['cd']))
rows = np.array(rows)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for i, (name, j) in enumerate([('Cl', 0), ('Cd', 2)]):
    ax[i].scatter(rows[:, j+1], rows[:, j], s=18)
    lo, hi = rows[:, j:j+2].min(), rows[:, j:j+2].max()
    ax[i].plot([lo, hi], [lo, hi], 'k--', lw=1)
    ax[i].set_xlabel(f'CFD {name}'); ax[i].set_ylabel(f'pred {name}'); ax[i].set_title(name)
plt.tight_layout(); plt.show()


## 10 · Save a report & next steps

- A standalone HTML report of the solve:


In [ ]:
report = res.save_report(os.path.join(CKPT_DIR, 'sample_report.html'))
print('report:', report)


**Scale up for real accuracy:**

- `TASK='full'`, `n_train=800`, `n_val=200`.
- Bigger backbone: `ModelConfig(name='fno', width=64, modes=24, n_layers=5)`
  (or `name='transformer'` for the Transolver-style physics-attention model).
- More epochs (150–300) and a higher `resolution` (192/256) on an A100.
- The checkpoint on Drive can be reloaded any time with
  `NeuroForgeEngine.from_checkpoint(ckpt_path)`.

See `docs/paper/neuroforge_cfd.md` for the method and `docs/ROADMAP.md` for what's next.
